# WorldPop — covariates: VIIRS night-time lights (live)

The 54 WorldPop **covariate** layers (nightlights, slope, elevation, distances, built-up, …) are each a selectable product. Here we pull the VIIRS night-time lights (`cviirs`) for Kenya and crop to a bbox.

## Setup

Consolidate the imports up front. `earthlens.worldpop` provides the covariate `Catalog`; `EarthLens` is the unified download entry point; `pyramids` reads the written rasters back; `tempfile` gives us a scratch output directory.

In [ ]:
import tempfile

from pyramids.dataset import Dataset

from earthlens.core import EarthLens
from earthlens.worldpop import Catalog

## 1 · Find the covariate layers

Load the WorldPop catalog and filter its products down to the `covariates` REST family. Counting them confirms the full covariate set is available before we request a specific layer.

In [ ]:
cat = Catalog()
covariates = [
    p for p in cat.available_products() if cat.get(p).rest_alias == 'covariates'
]
print('covariate layers:', len(covariates))

### Describe the nightlights product

`describe()` returns the metadata for a single product; here we read the human-readable description of the VIIRS night-time lights layer (`cviirs`) we are about to download.

In [ ]:
print('cviirs ->', cat.describe('cviirs')['description'])

## 2 · Download VIIRS nightlights for Kenya

WorldPop writes files to disk, so we point the download at a temporary directory. We request the `cviirs` covariate for Kenya (`KEN`) across 2012–2016 and crop to a small bbox around the equator.

In [ ]:
d = tempfile.mkdtemp()

### Build the request

Construct the `EarthLens` request on its own line so every parameter — the country AOI, the year window, the `%Y` date format, and the lat/lon crop limits — is easy to read and adjust.

In [ ]:
covariates_request = EarthLens(
    data_source='worldpop',
    variables=['cviirs'],
    aoi='KEN',
    start='2012',
    end='2016',
    fmt='%Y',
    lat_lim=[-1.0, 1.0],
    lon_lim=[36.0, 38.0],
    path=d,
)

### Run the download

`download()` mosaics and crops the source tiles per year and returns the list of written file paths. We print their names to confirm one raster per available VIIRS year.

In [ ]:
paths = covariates_request.download(progress_bar=False)
print('written:', sorted(p.name for p in paths))

## 3 · Open a result with pyramids

One cropped raster was written per available VIIRS year (2012–2016). We open the earliest one and inspect its CRS and shape to verify the crop.

In [ ]:
first_path = sorted(paths)[0]
ds = Dataset.read_file(first_path)
print('EPSG:', ds.epsg, 'shape:', ds.shape)